# Case study: Early economy of the Mirage league

The main analysis of historical leagues has been completed using a consistent analytical pipeline including data preparation, exploratory analysis, economic indicators, PCA, and clustering methods. However, during the course of the project, a new Path of Exile league, **Mirage**, was released on March 6. At the time of completing the analysis, only seven days of market data were available for this league.

Because the dataset for the Mirage league is still limited, it was not included in the main historical analysis to avoid methodological inconsistencies. Instead, it is analysed separately as a case study focusing on the early phase - **launch hype**- of league economies.

To ensure comparability, the same economic indicators used in the main analysis are calculated for the first six days of all leagues, including Mirage. This allows the early economic structure of the new league to be compared with the early-stage economies of previous leagues. The new league is then projected into the existing analytical framework using PCA and clustering methods, which makes it possible to determine whether its early market dynamics resemble any of the previously identified types of league economies.

This case study therefore serves as an exploratory extension of the project and demonstrates how the developed analytical framework can be applied to analyse the economic characteristics of newly released leagues.

### Importing libraries

In [18]:
import pandas as pd
from itables import show

### Importing data

In [16]:
#importing already processed datasets for items and players
items= pd.read_csv("C:/Users/user/Desktop/Data Science/PoeMarketAnalyser/data/poeninja/items_data_processed/high_conf_items_processed_day.csv", sep=";", low_memory=False)
players = pd.read_csv("C:/Users/user/Desktop/Data Science/PoeMarketAnalyser/data/poeninja/players_data_processed/players_processed_day.csv", sep=";", low_memory=False)

#imporing new datasets for items and players
items_mirage= pd.read_csv("C:/Users/user/Desktop/Data Science/PoeMarketAnalyser/data/poeninja/items_data_raw/Mirage.items.csv", sep=";", low_memory=False)
players_mirage= pd.read_csv("C:/Users/user/Desktop/Data Science/PoeMarketAnalyser/data/poeninja/players_data_raw/players_mirage.csv", sep=";", low_memory=False)

## Minimal cleaning of new datasets
Preparing the data only doing what is needed for the case study analysis

### Players new data

In [12]:
# add Day column
players_mirage["Date"] = pd.to_datetime(players_mirage["Date"])
league_start = players_new.groupby("League")["Date"].transform("min")
# league day
players_mirage["Day"] = (players_mirage["Date"]-league_start).dt.days + 1
players_mirage

,League,Date,Players,Day
0,Mirage,2026-03-06,190653.0,1
1,Mirage,2026-03-07,158206.0,2
2,Mirage,2026-03-08,156804.0,3
3,Mirage,2026-03-09,135034.0,4
4,Mirage,2026-03-10,127003.0,5
5,Mirage,2026-03-11,120098.0,6
6,Mirage,2026-03-12,113917.0,7


### Items new data

In [19]:
show(items_mirage)

Loading ITables v2.7.0 from the internet... (need help?)


## Merging old and new datasets

## League economic indicators - for 7 first days of each league

### Item data aggregation

In [10]:
league_market = (items.groupby("League").agg(avg_price=("Value","mean"),median_price=("Value","median"),unique_items=("Name","nunique"),total_market_value=("Value","sum")).reset_index())
avg_daily_listings = (items.groupby(["League","Day"]).size().groupby("League").mean().reset_index(name="avg_daily_listings"))

league_market = league_market.merge(avg_daily_listings, on="League")
league_market

,League,avg_price,median_price,unique_items,total_market_value,avg_daily_listings
0,Affliction,166.311469,10.00,3927,3.591566e+08,19632.200000
1,Ancestor,93.457264,8.60,6271,2.428879e+08,23626.536364
2,Keepers,126.298868,4.00,5862,2.717724e+08,17494.471545
3,Mercenaries,200.063168,10.00,4238,5.684217e+08,20738.766423
4,Necropolis,99.574644,7.75,4118,2.149757e+08,18452.478632
5,Settlers,130.357415,9.00,4224,7.041004e+08,16931.996865


### Players data aggregation

In [19]:
league_players = (players.groupby("League").agg(mean_daily_players=("Players","mean")).reset_index())
league_players

,League,mean_daily_players
0,Affliction,49709.944954
1,Ancestor,44956.522936
2,Keepers,33186.631148
3,Mercenaries,36416.051471
4,Necropolis,32586.034483
5,Settlers,29170.198113


### Market Value Concentration

In [17]:
#top 1% item value share
def market_concentration(df):
    total_value = df["Value"].sum()
    top_n = int(len(df) * 0.01)
    top_value = df.nlargest(top_n, "Value")["Value"].sum()

    return pd.Series({
        "top_value_share": top_value / total_value
    })

market_concentration = items.groupby("League").apply(market_concentration, include_groups=False).reset_index()
market_concentration

,League,top_value_share
0,Affliction,0.757199
1,Ancestor,0.571500
2,Keepers,0.764562
3,Mercenaries,0.775632
4,Necropolis,0.763378
5,Settlers,0.779128


### Merging datasets

In [22]:
league_economy = (league_market.merge(market_concentration, on="League").merge(league_players, on="League"))
league_economy

,League,avg_price,median_price,unique_items,total_market_value,avg_daily_listings,top_value_share,mean_daily_players
0,Affliction,166.311469,10.00,3927,3.591566e+08,19632.200000,0.757199,49709.944954
1,Ancestor,93.457264,8.60,6271,2.428879e+08,23626.536364,0.571500,44956.522936
2,Keepers,126.298868,4.00,5862,2.717724e+08,17494.471545,0.764562,33186.631148
3,Mercenaries,200.063168,10.00,4238,5.684217e+08,20738.766423,0.775632,36416.051471
4,Necropolis,99.574644,7.75,4118,2.149757e+08,18452.478632,0.763378,32586.034483
5,Settlers,130.357415,9.00,4224,7.041004e+08,16931.996865,0.779128,29170.198113


### Market indicators per player

In [24]:
league_economy["listings_per_player"] = (league_economy["avg_daily_listings"] / league_economy["mean_daily_players"])
league_economy["market_value_per_player"] = (league_economy["total_market_value"] / league_economy["mean_daily_players"])

league_economy

,League,avg_price,median_price,unique_items,total_market_value,avg_daily_listings,top_value_share,mean_daily_players,listings_per_player,market_value_per_player
0,Affliction,166.311469,10.00,3927,3.591566e+08,19632.200000,0.757199,49709.944954,0.394935,7225.045282
1,Ancestor,93.457264,8.60,6271,2.428879e+08,23626.536364,0.571500,44956.522936,0.525542,5402.727855
2,Keepers,126.298868,4.00,5862,2.717724e+08,17494.471545,0.764562,33186.631148,0.527154,8189.214196
3,Mercenaries,200.063168,10.00,4238,5.684217e+08,20738.766423,0.775632,36416.051471,0.569495,15609.096854
4,Necropolis,99.574644,7.75,4118,2.149757e+08,18452.478632,0.763378,32586.034483,0.566270,6597.172224
5,Settlers,130.357415,9.00,4224,7.041004e+08,16931.996865,0.779128,29170.198113,0.580455,24137.663243


In [25]:
#exporting dataset
league_economy.reset_index(drop=True).to_csv("C:/Users/user/Desktop/Data Science/PoeMarketAnalyser/data/poeninja/items_data_processed/league_economic_indicators.csv",sep=";",index=False)